In [ ]:
import os, subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
cores = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else os.cpu_count()
print('usable vCPU:', cores)

In [ ]:
!rm -rf /content/Chess
!git clone --branch perf/batched-mcts --single-branch --depth 1 https://github.com/TahaSalmani/Chess.git /content/Chess
%cd /content/Chess
!pip install -q python-chess zstandard scikit-learn tensorboard python-box==6.0.2 ensure pyyaml
!pip install -q -e .
!git log --oneline -1

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

SRC = '/content/drive/MyDrive/chess/trained_model.pth'   # <-- edit
DST = '/content/Chess/artifacts/prepare_torch_callbacks/best_model.pth'
if os.path.exists(SRC):
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copy2(SRC, DST)
    print('copied', SRC, '->', DST)
else:
    print('not found:', SRC, '| continuing with random weights')

In [ ]:
!python research/bench_mcts_split.py

In [ ]:
!python research/bench_mcts.py

In [ ]:
!cat params.yaml
!python src/Chess/pipeline/stage_10_Pytorch_RL.py 2>&1 | tee stage10.log

In [ ]:
!ls -la artifacts/rl_model artifacts/self_play_data
from google.colab import files
files.download('/content/Chess/artifacts/rl_model/rl_trained_model.pth')